# Deep Research Agent

Interactive notebook for running the deep research agent with real LLMs and search.

## Setup

In [1]:
import os

# Set your API key here or via environment variable
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# os.environ["TAVILY_API_KEY"] = "tvly-..."

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first"

In [ ]:
from langchain_anthropic import ChatAnthropic

from deep_research import (
    ClarificationNeeded, Complete, Config,
    DeepResearchAgent, DraftReady, GapReview, PlanReady, PlanReview,
)
from deep_research.events import ResearchUpdate
from deep_research.providers.duckduckgo import DuckDuckGoSearchProvider

fast_llm     = ChatAnthropic(model="claude-haiku-4-5-20251001")
powerful_llm = ChatAnthropic(model="claude-sonnet-4-6")

print("LLMs ready")

## Configure the agent

In [ ]:
config = Config(
    max_research_loops=2,
    breadth=3,                  # search queries per loop
    enable_clarification=True,  # set True to be prompted for clarifying questions
    enable_plan_review=True,    # set True to review/redirect the sub-question list
    enable_gap_review=True,     # set True to approve/redirect gap analysis
    enable_draft_review=True,   # set True to review draft before finalising
)

agent = DeepResearchAgent(
    fast_llm=fast_llm,
    powerful_llm=powerful_llm,
    search_provider=DuckDuckGoSearchProvider(),
    config=config,
)

print(f"Agent ready  (thread: {agent.thread_id})")

## Run a query

Change `QUERY` and re-run this cell.

In [ ]:
QUERY = "How do I set up a python venv for my new project"

result = None
print(f"Query: {QUERY}\n{'='*60}")

async for event in agent.astream(QUERY):
    if isinstance(event, ClarificationNeeded):
        print(f"\n[Clarification needed]")
        answers = {}
        for q in event.questions:
            answers[q] = input(f"  {q}: ")
        await agent.resume(answers)

    elif isinstance(event, PlanReview):
        print(f"\n[Plan review] {len(event.sub_questions)} sub-questions:")
        for sq in event.sub_questions:
            print(f"  • {sq.question}")
        resp = input("\n  Approve or give feedback (press Enter to approve): ")
        await agent.resume(resp or "approve")

    elif isinstance(event, PlanReady):
        # Emitted only when enable_plan_review=False (informational)
        print(f"\n[Plan] {len(event.sub_questions)} sub-questions:")
        for sq in event.sub_questions:
            print(f"  • {sq.question}")

    elif isinstance(event, ResearchUpdate):
        print(f"\n[Loop {event.loop_count}] {event.sources_count} sources, {event.findings_count} findings")

    elif isinstance(event, GapReview):
        print(f"\n[Gap review] Missing: {event.gaps}")
        resp = input("  Approve gaps / provide custom queries: ")
        await agent.resume(resp or "approve")

    elif isinstance(event, DraftReady):
        print(f"\n[Draft ready] ({len(event.draft)} chars)")
        resp = input("  Approve or give feedback: ")
        await agent.resume(resp or "approve")

    elif isinstance(event, Complete):
        result = event.result
        m = result.metadata
        print(f"\n{'='*60}")
        print(f"Done — {m['loops_run']} loops, {m['total_sources']} sources, {m['elapsed_seconds']}s")

# drain pending async callbacks
import asyncio; await asyncio.sleep(0)

## Report

In [12]:
from IPython.display import Markdown, display

if result:
    display(Markdown(result.final_report))
else:
    print("No result yet — run the query cell first.")

## What Are Python Virtual Environments and Why They Matter

A Python virtual environment is an isolated directory that contains its own independent set of Python packages, separate from your system-wide Python installation. According to research findings on Python's built-in `venv` module, each virtual environment maintains its own package directory, which means that installing, updating, or removing a package in one project has no effect on any other project on your machine. This isolation is the core purpose of the tool: rather than sharing a single global Python environment across all your work, each project gets its own contained space to operate within.

The practical importance of this isolation becomes clear when managing multiple projects simultaneously. As noted in research on virtual environment best practices, version conflicts are a common and frustrating problem when multiple projects share the same Python installation — for example, one project may require an older version of a library while another depends on a newer one. Virtual environments solve this by allowing developers to manage dependencies independently for each project. Best practices recommend creating a dedicated virtual environment for every new project and naming it consistently (commonly `venv` or `.venv`) to keep things organized and predictable across teams.

It is also worth understanding how `venv` fits within the broader ecosystem of similar tools. Research comparing Python environment managers notes that `venv` is built directly into Python 3 and serves as the standard, lightweight choice for most modern Python projects. Alternatives like `conda` offer broader capabilities — such as managing non-Python system-level packages — making them better suited for data science or machine learning work. Tools like `pipenv` combine environment management with dependency specification in a single workflow. For a straightforward new project, however, `venv` is widely recognized as the recommended starting point due to its simplicity and the fact that it requires no additional installation.

In short, Python virtual environments are a foundational practice for keeping your projects clean, conflict-free, and reproducible — and `venv`, as Python 3's built-in solution, is the most accessible way to get started.

## Comparing venv, virtualenv, conda, and pipenv

Python developers have several tools available for managing virtual environments, each with distinct strengths and trade-offs. According to the research findings on python venv vs conda virtualenv pipenv comparison, **venv** is built directly into Python 3's standard library, making it the most accessible and recommended starting point for most modern projects. It creates lightweight, isolated environments with their own independent package directories, requiring no additional installation. **Virtualenv**, by contrast, is an older third-party alternative that predates venv and remains relevant primarily for projects that must support Python 2, though for Python 3 development, venv is generally preferred.

Where venv and virtualenv focus narrowly on Python package isolation, **conda** takes a broader approach. As noted in the research, conda is part of the Anaconda and Miniconda distributions and is capable of managing system-level dependencies and non-Python packages—functionality that venv simply does not offer. This makes conda particularly well-suited for data science, machine learning, and scientific computing workflows, where projects frequently depend on compiled libraries or tools outside the Python ecosystem. However, this additional power comes with greater overhead, making conda a heavier-weight solution than venv for straightforward Python development.

**Pipenv** occupies a different niche from the other tools. According to the research, pipenv was created specifically to address shortcomings in virtualenv, particularly around distinguishing between direct project dependencies and their transitive dependencies, as well as separating development and production environments. It combines virtual environment management with dependency specification in a single workflow, offering a more streamlined experience for teams that prioritize reproducibility and clear dependency hierarchies. For most standard Python projects, however, venv paired with a `requirements.txt` file generated via `pip freeze` provides a straightforward and sufficient solution without the added complexity.

In summary, **venv is the recommended default for most Python 3 projects**, while conda is the better choice for data science and non-Python dependencies, and pipenv or virtualenv may be preferred in specialized scenarios involving complex dependency management or legacy Python 2 support.

## Setting Up venv on Windows, macOS, and Linux

Python's built-in `venv` module provides a straightforward, platform-consistent method for creating isolated virtual environments. According to the research findings, the core command for creating a virtual environment is identical across all three major operating systems: `python -m venv path/to/venv/`. Because `venv` is included in Python 3's standard library, no additional installation is required, making it the recommended starting point for most modern Python projects. A common convention is to run this command in your project's root directory—for example, `python -m venv venv` or `python -m venv .venv`—so that the environment lives alongside your project files and is easy to locate.

Once the environment is created, activation differs by platform. On macOS and Linux, the activation command is `source [directory_name]/bin/activate`, while Windows uses a separate activation script located in the environment's directory. After activation, your terminal prompt will update to display the environment name, confirming that subsequent `pip` commands will install packages into the isolated environment rather than your system-wide Python installation. This isolation is central to the tool's value: as the research notes, virtual environments allow developers to manage dependencies independently across projects, preventing the version conflicts that arise when multiple projects share a single Python installation.

With the environment active, packages can be installed using the familiar `pip install package_name` command. To capture all installed dependencies for reproducibility, the research recommends running `pip freeze` to generate a `requirements.txt` file, which other developers—or your own future setup—can use to recreate the exact environment by running `pip install -r requirements.txt`. Additionally, best practices highlighted in the findings advise adding the virtual environment directory to your project's `.gitignore` file, using the GitHub Python `.gitignore` template as a standard reference, so that the environment folder is never accidentally committed to version control. To exit the environment when your work session is complete, running `deactivate` in the terminal returns you to the system Python context.

In summary, setting up a Python `venv` involves three consistent steps across all platforms—creating, activating, and managing packages within the environment—with only the activation command varying by operating system.

## Activating, Deactivating, and Managing Packages

Once a virtual environment has been created using `python -m venv venv`, the next essential step is activation, which tells your terminal session to use the isolated environment's Python interpreter and package directory rather than the system-wide installation. According to [https://python.land/virtual-environments/virtualenv], the activation command varies by operating system: on Linux and macOS, users run `source venv/bin/activate` in the terminal, while Windows relies on a separate activation script (typically `venv\Scripts\activate.bat`). Upon successful activation, the environment's name will appear in the terminal prompt, providing a clear visual confirmation that subsequent Python and pip commands will operate within the isolated environment. This distinction is critical—without activation, any packages installed via pip will be added to the global Python installation, undermining the purpose of the virtual environment entirely.

With the environment active, package management becomes straightforward and contained. According to [https://python.land/virtual-environments/virtualenv], you can install any required package using the standard `pip install package_name` command, and all installed packages will be stored within the virtual environment's own directory, leaving the rest of the system unaffected. When you have finished working on a project or need to switch contexts, deactivating the environment is equally simple—running the `deactivate` command in the terminal restores the session to using the system Python installation.

A particularly valuable practice for team collaboration and project reproducibility is capturing all installed dependencies in a `requirements.txt` file. According to [https://python.land/virtual-environments/virtualenv], this is accomplished by running `pip freeze > requirements.txt`, which records every installed package and its exact version. Any developer who subsequently clones the project can then recreate the identical environment by running `pip install -r requirements.txt`, ensuring consistent behavior across different machines and development setups. Best practices also recommend adding the virtual environment directory (whether named `venv` or `.venv`) to the project's `.gitignore` file, as noted by [https://python.land/virtual-environments/virtualenv], to prevent the environment's contents—which can be large and are machine-specific—from being committed to version control.

In summary, the activation, deactivation, and package management workflow surrounding Python virtual environments provides developers with a reliable, reproducible, and collaborative foundation for managing project dependencies in complete isolation from the broader system.

## Dependency Management with requirements.txt

Once a virtual environment is activated, managing project dependencies effectively is a critical next step. According to [https://docs.python.org/3/library/venv.html], packages can be installed into the active environment using `pip install package_name`, ensuring they are confined to that project rather than installed system-wide. This isolation means that each project can maintain its own set of package versions without interfering with other projects or the system Python installation—a core advantage of using virtual environments in the first place.

The `requirements.txt` file is the standard mechanism for recording and sharing a project's dependencies. According to [https://pip.pypa.io/en/stable/user_guide/], running `pip freeze` within an active virtual environment outputs a list of all currently installed packages along with their exact version numbers. Redirecting this output to a file—`pip freeze > requirements.txt`—creates a snapshot of the environment that can be committed to version control alongside the project's source code. This file serves as a reproducible blueprint for the project's dependency state at any given time.

When another developer needs to set up the same environment—or when you return to a project on a new machine—the saved dependencies can be reinstalled with a single command: `pip install -r requirements.txt`. According to [https://realpython.com/python-virtual-environments-a-primer/], this workflow makes it straightforward to reproduce the exact environment later, ensuring consistency across different machines and development setups. It is worth noting that best practices recommend adding the virtual environment directory itself (commonly named `venv` or `.venv`) to the project's `.gitignore` file, so that only the `requirements.txt` file—not the entire environment folder—is tracked in version control.

In summary, combining `pip freeze` to generate a `requirements.txt` file with `pip install -r requirements.txt` to restore it provides a simple, reliable workflow for capturing and reproducing project dependencies across any development environment.

## Best Practices for Project Organization and Version Control

When organizing a Python project that uses a virtual environment, consistency and clarity in naming and structure are foundational. According to the research findings on virtual environment best practices, naming the environment directory uniformly—either `venv` or `.venv`—is widely recommended to make projects immediately recognizable to collaborators. Placing the virtual environment folder at the root of the project directory keeps all project-related files in one location, and using `python -m venv venv` in that root directory is the standard command to accomplish this across Windows, macOS, and Linux. This consistency ensures that anyone picking up the project can orient themselves quickly without needing to hunt for environment configuration files.

A critical and non-negotiable step in version control is excluding the virtual environment directory from your repository. As noted in the research, best practices call for adding the environment folder to your `.gitignore` file, preventing the often-large collection of installed packages from being committed to version control. The GitHub Python `.gitignore` template provides a standard reference for which Python artifacts—including virtual environment directories—should be excluded. Committing the virtual environment itself is unnecessary and counterproductive, since the environment is platform-specific and may not function correctly on another developer's machine or operating system.

Instead of committing the environment, the correct approach is to capture project dependencies in a `requirements.txt` file and commit that. According to the research, running `pip freeze` after installing all necessary packages outputs a precise list of dependencies and their versions, which can be saved to `requirements.txt`. Any developer cloning the repository can then recreate the identical environment by creating a fresh virtual environment and running `pip install -r requirements.txt`. This workflow ensures reproducibility across different machines and team members without bloating the repository. For projects with more complex dependency needs—such as those distinguishing between development and production dependencies—tools like `pipenv` offer additional structure, though `venv` combined with a well-maintained `requirements.txt` is sufficient for most standard Python projects.

In summary, the key takeaway for project organization and version control is to store a `requirements.txt` file in your repository rather than the virtual environment itself, using `.gitignore` to exclude the environment directory and ensuring any collaborator can reliably reconstruct the project's dependencies from scratch.

## Sources

- [reddit.com-1] A Quick Guide on How to Setup a Python Virtual Environment ... - Reddit — https://www.reddit.com/r/Python/comments/ju3ybo/a_quick_guide_on_how_to_setup_a_python_virtual/
- [stackoverflow.com-1] How can I set up a virtual environment for Python in Visual Studio Code? — https://stackoverflow.com/questions/54106071/how-can-i-set-up-a-virtual-environment-for-python-in-visual-studio-code
- [quora.com-1] How do you install a Python environment? - Quora — https://www.quora.com/How-do-you-install-a-Python-environment
- [quora.com-2] What is the best way to install an environment for Python? - Quora — https://www.quora.com/What-is-the-best-way-to-install-an-environment-for-Python
- [matthewhard.com-1] Set Up a Python venv on Any OS (Beginner Guide) — https://matthewhard.com/how-to-set-up-a-python-virtual-environment-venv-on-windows-macos-and-linux-for-absolute-beginners
- [pythonguis.com-1] Python Virtual Environments: How to Create, Use & Manage venv — https://www.pythonguis.com/tutorials/python-virtual-environments/
- [dev.to-1] Python Virtual Environments: Why You Need Them and How to Use Them — https://dev.to/lovestaco/python-virtual-environments-why-you-need-them-and-how-to-use-them-1o1e
- [towardsdatascience.com-1] Python Virtual Environments: Why and When Should You Use Them? — https://towardsdatascience.com/python-virtual-environments-why-and-when-should-you-use-them-be57b0c0323d/
- [realpython.com-1] Python Virtual Environments: A Primer - Real Python — https://realpython.com/python-virtual-environments-a-primer/
- [pythonguides.com-1] Virtual Environments in Python — https://pythonguides.com/virtual-environments-python/
- [reddit.com-2] virtualenv, vs pipenv, vs conda? Is one superior to the others? If not, under ... — https://www.reddit.com/r/learnpython/comments/or1qwh/virtualenv_vs_pipenv_vs_conda_is_one_superior_to/
- [codesolid.com-1] Conda vs. Pip, Venv, and Pyenv – Simplicity Wins - CodeSolid — https://codesolid.com/conda-vs-pip/
- [medium.com-1] Pipenv vs virtualenv vs conda environment | by Krishna Regmi — https://medium.com/@krishnaregmi/pipenv-vs-virtualenv-vs-conda-environment-3dde3f6869ed
- [stackoverflow.com-2] What is the difference between venv, pyvenv, pyenv, virtualenv ... — https://stackoverflow.com/questions/41573587/what-is-the-difference-between-venv-pyvenv-pyenv-virtualenv-virtualenvwrappe
- [medium.com-2] Understanding when and how to use Conda, Pipenv, Virtualenv, Pip, and ... — https://medium.com/@maheshkarthu/understanding-when-and-how-to-use-conda-pipenv-virtualenv-pip-and-poetry-is-crucial-for-2a518a951945
- [docs.python.org-1] venv — Creation of virtual environments — Python 3.14.5 ... — https://docs.python.org/3/library/venv.html
- [3schools.com-1] Python Virtual Environment - venv - W3Schools — https://www.w3schools.com/python/python_virtualenv.asp
- [gist.github.com-1] Set up Python 3 and new virtual environment - GitHub Gist — https://gist.github.com/MichaelCurrin/3a4d14ba1763b4d6a1884f56a01412b7
- [medium.com-3] Setting up a Python Virtual Environment on Windows ... - Medium — https://medium.com/@rodolfo.antonio.sep/setting-up-a-python-virtual-environment-on-windows-september-2023-aa9a25f856f9
- [python.land-1] Python venv: How To Create, Activate, Deactivate, And Delete — https://python.land/virtual-environments/virtualenv
- [packaging.python.org-1] Install packages in a virtual environment using pip and venv — https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/
- [virtualenv.pypa.io-1] Getting started - virtualenv — https://virtualenv.pypa.io/en/latest/tutorial/getting-started.html
- [gist.github.com-2] Python venv virtual environment cheat sheet - GitHub Gist — https://gist.github.com/ryanbehdad/858b47b54be441a684efb7ae6ca98a75
- [techntuts.com-1] The Complete Guide to Python Virtual Environments, activate, deactivate ... — https://www.techntuts.com/story/The-Complete-Guide-Python-Virtual-Environments-activate-deactivate-Requirements.txt
- [github.com-1] gitignore/Python.gitignore at main · github/gitignore · GitHub — https://github.com/github/gitignore/blob/main/Python.gitignore
- [medium.com-4] Python Virtual Environments: A Beginner's Guide to venv - Medium — https://medium.com/@thomasjay200/python-virtual-environments-a-beginners-guide-to-venv-61af170893b7
- [spletzer.com-1] A No-Nonsense Guide to Setting Up Python Environments - Ryan Spletzer — https://www.spletzer.com/2024/04/a-no-nonsense-guide-to-setting-up-python-environments/
- [docs.python.org-2] 12. Virtual Environments and Packages — Python 3.14.5 documentation — https://docs.python.org/3/tutorial/venv.html
- [apidog.com-1] How to Activate Python Virtual Environments (venv): Complete Guide — https://apidog.com/blog/python-venv/
- [frankcorso.dev-1] Setting Up Your Python Environment With Venv and requirements.txt - Frank's Blog — https://frankcorso.dev/setting-up-python-environment-venv-requirements.html
- [youtube.com-1] How to choose an environment virtualization method – Venv vs. Conda — https://www.youtube.com/watch?v=q9iFnMEEkwQ
- [inventivehq.com-1] Python Virtual Environments: venv, virtualenv, conda, and pyenv... — https://inventivehq.com/blog/python-virtual-environments-guide

## Sources

In [7]:
if result:
    for url, src in result.sources.items():
        print(f"[{src.relevance_score:.2f}] {src.title}")
        print(f"        {url}")

[0.00] Use different Python version with virtualenv - Stack Overflow
        https://stackoverflow.com/questions/1534210/use-different-python-version-with-virtualenv
[0.00] Python Virtual Environment “venv” Cheat Sheet
        https://blog.finxter.com/python-virtual-environments-with-venv-a-step-by-step-guide/
[0.00] Install packages in a virtual environment using pip and venv -
        https://packaging.python.org/en/latest/guides/installing-using-pip-and-virtual-environments/
[0.00] Mastering Python Development: A Step-by-Step Guide to Setting
        https://biplov.dev/mastering-python-development-a-step-by-step-guide-to-setting-up-virtual-environments-on-your-pc/
[0.00] Python `venv` - LearnWagtail.com
        https://learnwagtail.com/docs/guides/installation/venv/
[0.00] Best Practices for Managing Python Dependencies
        https://www.geeksforgeeks.org/python/best-practices-for-managing-python-dependencies/
[0.00] Best Practices for Using pip - DataScienceBase
        https://w

## Inspect state

In [8]:
state = await agent.aget_state()
print(f"loops run  : {state['research_loop_count']}")
print(f"findings   : {len(state['findings'])}")
print(f"sources    : {len(state['sources'])}")
print(f"is_sufficient: {state['is_sufficient']}")

loops run  : 2
findings   : 11
sources    : 52
is_sufficient: True
